# One sequence, three equivalent computations

How much does weakening past-state carry change a fixed filter's final signed readout?

Implement recurrent, quadratic and chunked SSD, forecast a number, then inspect an effect curve and state heatmaps. This exact small filter connects to [5.4 Mamba-2 and State-Space Duality](../../../chapter5_modern_architectures/exercises/part4_mamba2_ssd/5.4_Mamba-2_and_State-Space_Duality_exercises.ipynb). It tests state transport; it does not reproduce the historical trained recall, 130M checkpoint or CUDA results.

The public case was fixed before measurement. No outcome sign or monotonic trend is promised. Viewing any case output before your commitment counts as prior exposure; local records cannot certify that you have never seen it.

In [ ]:
from pathlib import Path
import sys
import torch as t
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'mastery_hub').is_dir())
if str(root) not in sys.path:sys.path.insert(0,str(root))
from mastery_hub.units.ssd_prediction import lesson
from mastery_hub.units.ssd_prediction.lesson import validate_inputs
from IPython.display import display
import math
t.set_num_threads(1)
# CPU float64 is explicit in every method input; no target runs on import.


## Public inputs and prediction

X is [time,head], shared B/C are [time,state], and initial state is [head,state]. Read **after** writing:

`state[t,h,n] = gamma*exp(A[h])*state[t-1,h,n]+X[t,h]*B[t,n]`

`Y[t,h] = sum_n C[t,n]*state[t,h,n]`; `r[t]=.7*Y[t,0]-.4*Y[t,1]`.

T=12, H=2, N=2, zero initial state, CPU float64, one thread. A=log([.82,.94]). Primary target: **r[11](gamma=.65) minus r[11](gamma=1)**, signed filter-output units; there is no [-1,1] cap. Chunk size5 is an equivalent factorization, not a changed model.

The curve uses gamma [0,.25,.5,.65,.85,1]. Transfer changes only A→A/2, requires a separate forecast and commitment, and is never a primary control.

In [ ]:
# Inspect raw public inputs only; this performs no SSD or readout evaluation.
public_inputs=lesson._case()
public_inputs

## Implement ssd_recurrent

At each real token decay the previous state, add the outer-product write, then read. Return every output and every real-token state.

<details><summary>Help</summary>Use the different T3 practice fixture. Expected feedback is “Practice recurrent passed”. Reference methods become available only after a commitment; edits afterwards require a new commitment.</details>

In [ ]:
RECURRENT_SOURCE = '''
def ssd_recurrent(X, A, B, C, initial, gamma):
    raise NotImplementedError("Implement ssd_recurrent before freezing")
'''
namespace=lesson.method_namespace(RECURRENT_SOURCE,complete=False)
print(lesson.practice_test(namespace,'recurrent'))

Expected: Practice recurrent passed (different inputs). No public-case method is called here.

## Implement ssd_quadratic

For t>=s the write weight is exp(sum of effective log transitions over s+1..t). The diagonal is one. Handle gamma0 explicitly: no log(0) or0*(-inf). Include initial-state decay separately.

<details><summary>Help</summary>Use the different T3 practice fixture. Expected feedback is “Practice quadratic passed”. Reference methods become available only after a commitment; edits afterwards require a new commitment.</details>

In [ ]:
QUADRATIC_SOURCE = '''
def ssd_quadratic(X, A, B, C, initial, gamma):
    raise NotImplementedError("Implement ssd_quadratic before freezing")
'''
namespace=lesson.method_namespace(QUADRATIC_SOURCE,complete=False)
print(lesson.practice_test(namespace,'quadratic'))

Expected: Practice quadratic passed (different inputs). No public-case method is called here.

## Implement ssd_chunked

Use a causal quadratic diagonal block per chunk plus the decayed incoming boundary state. Loop over chunks, not tokens. Fold gamma into real transitions BEFORE identity padding: padding has decay1 and zero writes/reads. Crop every returned array to real tokens.

<details><summary>Help</summary>Use the different T3 practice fixture. Expected feedback is “Practice chunked passed”. Reference methods become available only after a commitment; edits afterwards require a new commitment.</details>

In [ ]:
CHUNKED_SOURCE = '''
def ssd_chunked(X, A, B, C, initial, gamma, chunk_size):
    raise NotImplementedError("Implement ssd_chunked before freezing")
'''
namespace=lesson.method_namespace(CHUNKED_SOURCE,complete=False)
print(lesson.practice_test(namespace,'chunked'))

Expected: Practice chunked passed (different inputs). No public-case method is called here.

## Implement readout

Contract the head output with the frozen readout weights. Preserve signed values.

<details><summary>Help</summary>Use the different T3 practice fixture. Expected feedback is “Practice readout passed”. Reference methods become available only after a commitment; edits afterwards require a new commitment.</details>

In [ ]:
READOUT_SOURCE = '''
def readout(outputs, weights):
    raise NotImplementedError("Implement readout before freezing")
'''
namespace=lesson.method_namespace(READOUT_SOURCE,complete=False)
print(lesson.practice_test(namespace,'readout'))

Expected: Practice readout passed (different inputs). No public-case method is called here.

## Implement assemble_result

Convert the nine actual output/state/readout tensors to JSON-native lists under recurrent/quadratic/chunked. Do not recompute or fabricate measurements.

<details><summary>Help</summary>Use the different T3 practice fixture. Expected feedback is “Practice assembly passed”. Reference methods become available only after a commitment; edits afterwards require a new commitment.</details>

In [ ]:
ASSEMBLY_SOURCE = '''
def assemble_result(ry, rs, rr, qy, qs, qr, cy, cs, cr):
    raise NotImplementedError("Implement assemble_result before freezing")
'''
namespace=lesson.method_namespace(ASSEMBLY_SOURCE,complete=False)
print(lesson.practice_test(namespace,'assembly'))

Expected: Practice assembly passed (different inputs). No public-case method is called here.

## Exposed DEV ladder — trace the carry before forecasting (8–12 minutes)
Use the existing T3 fixture below, not the T12 target. Hand-trace token 0:
head-0 initial [.5,−.25], decay .5*gamma, write [1,0], read [1,1], so Y[0,0]=1+.125*gamma.
For head 1, Y[0,1]=2+.5625*gamma. Predict both at gamma=.4, then compare gamma=0.
Choose **A:** stronger carry must increase every signed output; **B:** carry transports state,
but signed writes and read directions determine the output. Name a paired trajectory that
could rule A out. No target outcome is used here. Numeric confidence is an **unscored self-rating**.

In [ ]:
dev_forecast_first=[1.05,2.225]
dev_self_rating=.7  # unscored self-rating
dev_explanation='B'
dev_disconfirming_observation='More carry can lower a later signed output.' 

In [ ]:
import matplotlib.pyplot as plt
# Public T3 practice literals; no public-target runner or _case call.
practice_X,practice_A,practice_B,practice_C,practice_initial=lesson.practice_fixture()
dev_source='\n\n'.join([RECURRENT_SOURCE,QUADRATIC_SOURCE,CHUNKED_SOURCE,READOUT_SOURCE,ASSEMBLY_SOURCE])
dev_ns=lesson.method_namespace(dev_source)
dev_outputs={};dev_states={}
for gamma in (0.,.4):
    y,state=dev_ns['ssd_recurrent'](practice_X,practice_A,practice_B,practice_C,practice_initial,gamma)
    dev_outputs[gamma]=y;dev_states[gamma]=state
    cy,cs=dev_ns['ssd_chunked'](practice_X,practice_A,practice_B,practice_C,practice_initial,gamma,2)
    t.testing.assert_close(cy,y);t.testing.assert_close(cs,state)
zero_y,zero_s=dev_ns['ssd_recurrent'](t.zeros_like(practice_X),practice_A,practice_B,practice_C,t.zeros_like(practice_initial),.4)
fig,axes=plt.subplots(1,2,figsize=(11,3),layout='constrained')
for head,ax in enumerate(axes):
    for gamma,y in dev_outputs.items(): ax.plot(range(3),y[:,head].tolist(),'o-',label='gamma='+str(gamma))
    ax.plot(range(3),zero_y[:,head].tolist(),'--',label='Zero write + initial control')
    ax.set(xlabel='DEV token',ylabel='Signed output',title='DEV head '+str(head),xticks=range(3));ax.legend()
display(fig)
t.testing.assert_close(dev_outputs[.4][0],t.tensor([1.05,2.225],dtype=t.float64))
print('DEV first-token absolute forecast error:',(dev_outputs[.4][0]-t.tensor(dev_forecast_first,dtype=t.float64)).abs().tolist())
assert zero_s.count_nonzero()==0
print('DEV output changes:',(dev_outputs[.4]-dev_outputs[0.]).tolist())

### Worked DEV reflection / self-check
The first-token prediction [1.05,2.225] follows directly from read-after-write. Later head-0
output falls from .0625 to .034 at token 2: the read [.5,.25] mixes transported
state with a new signed write. That observed decrease rules A out even though
initial carry raised token 0. A good reflection cites one increase, one decrease, the unchanged
quadratic/chunked factorization check, and the inert zero-write/zero-initial control.
A wrong monotonic forecast is a reason to revise the readout model, not tune the target.

### Try It Yourself — changed-condition DEV transfer
Freeze one change: negate ONLY the last-token read direction C[2]. Predict the last output
and the states before running. Explanation B predicts opposite last outputs with identical
states; a story that output sign directly measures stored-memory sign predicts otherwise.

In [ ]:
dev_transfer_C=practice_C.clone();dev_transfer_C[-1]*=-1
dev_transfer_forecast=(-dev_outputs[.4][-1]).clone()
dev_transfer_y,dev_transfer_s=dev_ns['ssd_recurrent'](practice_X,practice_A,practice_B,dev_transfer_C,practice_initial,.4)
t.testing.assert_close(dev_transfer_y[-1],-dev_outputs[.4][-1])  # Method invariant, not a forecast gate.
print('DEV last-token absolute forecast error:',(dev_transfer_y[-1]-dev_transfer_forecast).abs().tolist())
t.testing.assert_close(dev_transfer_s,dev_states[.4])
t.testing.assert_close(dev_transfer_y[:-1],dev_outputs[.4][:-1])
fig,ax=plt.subplots(figsize=(6,3),layout='constrained')
ax.plot(range(3),dev_outputs[.4][:,0].tolist(),'o-',label='Original read')
ax.plot(range(3),dev_transfer_y[:,0].tolist(),'o-',label='Last read negated')
ax.plot(range(3),zero_y[:,0].tolist(),'--',label='Inert control')
ax.set(xlabel='DEV token',ylabel='Head-0 signed output',title='Observed DEV read-direction transfer',xticks=range(3));ax.legend();display(fig)
print('DEV transfer last output:',dev_transfer_y[-1].tolist())
dev_revision=''  # Explain how identical states can give opposite last outputs.

## Commit before any public-case result

Write a signed numerical forecast, an unscored confidence self-rating in [0,1], and causal reasoning. Only absolute point-forecast error is assessed; no probabilistic calibration follows. State exposure honestly: unknown, seen, or self_reported_not_seen. Freeze binds ordered inputs/layout/dtype, exact method/helper/test source and CPU runtime. Freeze/commit do not run the target. Change code or runtime? Freeze and commit again.

In [ ]:
prediction=None
confidence=None  # unscored self-rating in [0,1], not a probability
reasoning=''
prior_exposure='unknown'

In [ ]:
method_source="\n\n".join([RECURRENT_SOURCE,QUADRATIC_SOURCE,CHUNKED_SOURCE,READOUT_SOURCE,ASSEMBLY_SOURCE])
identity=lesson.freeze_inputs(method_source)
# Fill the forecast cell, then explicitly commit:
commitment=lesson.commit_prediction(identity,method_source,prediction,confidence,reasoning,prior_exposure)

## Reveal, then measure once

After commitment you may inspect reference code with `lesson.reveal_methods(commitment)`. It is diagnostic help; changing the committed implementation requires a new forecast/commitment. Only then run the public case and its controls. Keep any `ExperimentFailure.record_text`; it is an unscored failure, never silently retried.

In [ ]:
try:
    receipt=lesson.run_experiment(commitment,method_source)
except lesson.ExperimentFailure as error:
    failure_record=error.record_text
    print('Keep failure_record and export it; diagnose before any retry.')
    raise
lesson.plot_receipt(receipt)
# lesson.export_record(commitment, '/tmp/ssd-commitment.json')
# lesson.export_record(receipt, '/tmp/ssd-receipt.json')

## Reflect and forecast the single-factor transfer

Explain your forecast error using actual state/readout trajectories and the controls. No monotonicity follows merely from stronger carry because signed writes and read directions can cancel. The transfer changes only A→A/2; keep all other inputs and the same signed metric. Reflection and replay compute no new experiment.

In [ ]:
# attempt=lesson.reflect(receipt, explanation='',transfer_prediction=None,confidence=None,reasoning='')
# transfer_identity=lesson.freeze_inputs(method_source,variant='transfer')
# transfer_commitment=lesson.commit_transfer(attempt,transfer_identity,method_source,prior_exposure)
# transfer_receipt=lesson.run_experiment(transfer_commitment,method_source)
# lesson.plot_receipt(transfer_receipt)
# lesson.export_attempt(attempt, '/tmp/ssd-reflection.json')

## Try It Yourself — a different practice filter

Change gamma, chunk size or an input in the T3 fixture and compare equivalent methods. This fixture is different from the public case, so its values do not reveal your primary or transfer answer. The orthogonal state-basis and split-cache tests also use this fixture.

In [ ]:
# This practice-only play never calls the public-case runner.
practice_X,practice_A,practice_B,practice_C,practice_initial=lesson.practice_fixture()
practice_gamma=.4
practice_chunk_size=2
practice_namespace=lesson.method_namespace(method_source)
py,ps=practice_namespace['ssd_chunked'](practice_X,practice_A,practice_B,practice_C,practice_initial,practice_gamma,practice_chunk_size)
import matplotlib.pyplot as plt
plt.plot(py.numpy());plt.xlabel('Practice token');plt.ylabel('Practice head output')

## What the controls establish

After commitment, all three computations are checked against a scalar nested-loop oracle. Different chunk sizes, nondivisible padding, future perturbations, zero writes, gamma0, linearity and coherent head/readout swaps test algorithm correctness. Split-state/nonzero-initial-state and orthogonal basis controls use distinct practice inputs. Their success does not establish language-model memory or speed. This companion has no sampling, training, model loading, download, GPU or released-model claim. Historical 5.4 evidence stays in the full lesson.

Exports use private exclusive files; replay plots stored arrays without rerunning methods. Local hashes catch accidental drift, not hostile local editing or false exposure reports. Keep failed records and diagnose them before any retry.